In [0]:
import requests
from datetime import datetime
import pandas as pd

# Fetch Bitcoin price from CoinGecko
url = "https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd"

response = requests.get(url)
data = response.json()

# Create a Spark-friendly DataFrame
df = pd.DataFrame([{
    "timestamp": datetime.now(),
    "bitcoin_price": data['bitcoin']['usd']
}])

df


,timestamp,bitcoin_price
0,2025-10-01 22:40:05.134675,117780


In [0]:
display(df)

timestamp,bitcoin_price
2025-10-01T22:40:05.134Z,117780


In [0]:
# Convert Pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(df)

# Show the Spark DataFrame
spark_df.show()


+--------------------+-------------+
|           timestamp|bitcoin_price|
+--------------------+-------------+
|2025-10-01 22:40:...|       117780|
+--------------------+-------------+



In [0]:
# Save Spark DataFrame as a Delta table (CE-friendly)
spark_df.write.format("delta").mode("append").saveAsTable("bitcoin_prices")

# Query the table to check
spark.sql("SELECT * FROM bitcoin_prices").show()


+--------------------+-------------+
|           timestamp|bitcoin_price|
+--------------------+-------------+
|2025-10-01 22:40:...|       117780|
+--------------------+-------------+



In [0]:
import requests
from datetime import datetime
import pandas as pd

# Fetch live Bitcoin price
url = "https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd"
response = requests.get(url)
data = response.json()

# Convert to Spark DataFrame
df = pd.DataFrame([{
    "timestamp": datetime.now(),
    "bitcoin_price": data['bitcoin']['usd']
}])

spark_df = spark.createDataFrame(df)

# Append to Delta table
spark_df.write.format("delta").mode("append").saveAsTable("bitcoin_prices")

print("New Bitcoin price inserted:", df)


New Bitcoin price inserted:                    timestamp  bitcoin_price
0 2025-10-01 22:50:50.845345         117839


In [0]:
# Read all data from the Delta table
bitcoin_df = spark.table("bitcoin_prices")
bitcoin_df.show()


+--------------------+-------------+
|           timestamp|bitcoin_price|
+--------------------+-------------+
|2025-10-01 22:58:...|       117891|
|2025-10-01 22:58:...|       117887|
|2025-10-01 22:50:...|       117839|
|2025-10-01 22:40:...|       117780|
+--------------------+-------------+



In [0]:
# Load all Bitcoin price data from your Delta table
bitcoin_df = spark.table("bitcoin_prices")
bitcoin_df.show()


+--------------------+-------------+
|           timestamp|bitcoin_price|
+--------------------+-------------+
|2025-10-01 22:58:...|       117891|
|2025-10-01 22:58:...|       117887|
|2025-10-01 22:50:...|       117839|
|2025-10-01 22:40:...|       117780|
+--------------------+-------------+



In [0]:
spark.sql("""
SELECT 
    DATE(timestamp) AS date, 
    AVG(bitcoin_price) AS avg_price, 
    MAX(bitcoin_price) AS max_price
FROM bitcoin_prices
GROUP BY DATE(timestamp)
ORDER BY date
""").show()


+----------+---------+---------+
|      date|avg_price|max_price|
+----------+---------+---------+
|2025-10-01|117849.25|   117891|
+----------+---------+---------+



In [0]:
display(bitcoin_df)


timestamp,bitcoin_price
2025-10-01T22:58:31.640Z,117891
2025-10-01T22:58:21.492Z,117887
2025-10-01T22:50:50.845Z,117839
2025-10-01T22:40:05.134Z,117780


Databricks visualization. Run in Databricks to view.